
# Tutorial on 0D calibration

This work is part of the Master Course for Biomechanics of the Cardiovascular System, offered in EPFL and taught by Professor Nikolaos Stergiopulos (nikolaos.stergiopulos@epfl.ch).
The material has been adapted for use in VITAL's first training school in Delft. Use of the material is permitted exclusively for the purposes of this workshop.

In [ ]:
from utilities.utils_tutorial1 import * #load_subject_data, calculate_input_impedance, calculate_characteristic_impedance, calculate_pwv, calculate_pulse_pressure, fc_wk3
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt

# Load the PWDB data
The Pulse Wave Data Base (PWDB) by Peter Charlton is described in this article https://journals.physiology.org/doi/full/10.1152/ajpheart.00218.2019. 

The pwdb_data.mat file, described here https://github.com/peterhcharlton/pwdb/wiki/pwdb_data.mat, is required to run the tutorials.

Please download the .mat file at the end of this page: https://zenodo.org/records/3275625 and add it to the 'data' folder.


In [ ]:
# Load the PWDB data
data = loadmat('data/pwdb_data.mat')
# Access the main data structure
pwdb_data = data['data']

In [ ]:
# Get the total number of subjects available
total_subjects = pwdb_data['waves'][0, 0]['P_AorticRoot'][0, 0].shape[1]
print(f"Total available subjects: {total_subjects}")

# Load the pressure and flow waveforms at the aortic root for one virtual subject

In [ ]:
# Example usage: Choose a subject ID dynamically
subject_id = int(input(f"Enter a subject ID (1 to {total_subjects}): "))
pressure, flow, area, HR, pvr, time = load_subject_data(subject_id, pwdb_data, 'AorticRoot')

Basic analysis of hemodynamic data
- Find the mean aortic pressure and flow

In [ ]:
print(f'Mean pressure is {np.mean(pressure):.1f} mmHg and mean flow is {np.mean(flow):.1f} ml/s')
plot_pressure_and_flow(time,pressure,flow)

- Find the total peripheral resistance

In [ ]:
SVR = np.mean(pressure)/np.mean(flow)
print(f'Total Peripheral Resistance: {SVR:.2f} mmHg·s/mL')
print(f'Total Peripheral Resistance (SI): {SVR*133/1e-6:.2e} Pa·s/m³')

# Calculate input impedance

In [ ]:
_ = calculate_input_impedance(time,pressure,flow)

# Calculate characteristic impedance

In [ ]:
_ = calculate_characteristic_impedance(time,pressure,flow)

# Estimate compliance using the pulse pressure method (2-element Windkessel)


In [ ]:
_ = calculate_compliance_using_PPM(time, pressure, flow, C_WK2_init=1.0)

- Now change the initial guess for the compliance (`C_WK2_init`). What trend do you observe for the estimated PP?

In [ ]:
_ = calculate_compliance_using_PPM(time, pressure, flow, C_WK2_init=0.95)

In [ ]:
Zc = calculate_characteristic_impedance(time, pressure, flow, show=False)

# Fit a 3-element Windkessel model

In [ ]:
_ = wk3(time, pressure, flow)

# Calculate the Pulse Wave Velocity (PWV)

In [ ]:
mean_area = np.mean(area)
rho = 1060  # blood density in kg/m³
PWV = calculate_pwv(rho, mean_area, Zc)

Extra:
- Fit an exponential to the diastolic portion of the aortic pressure waveform (find fiducial point according to Supplementary Table A3 in https://journals.physiology.org/doi/full/10.1152/ajpheart.00218.2019 )